In [17]:
from typing import List, Literal,TypedDict
from pydantic import BaseModel,Field
from pypdf import PdfReader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai  import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings 
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [18]:
load_dotenv()

True

In [22]:
#---------------LOAD DOCS-----------------------------------
#Helper function to replicate PyPDFLoader
def load_pdf_as_documents(file_path: str) -> list[Document]:
    try:
        reader = PdfReader(file_path)
        documents = []

        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            # Replicates LangChain's default metadata schema for loaders
            metadata = {"source": file_path, "page": i}
            documents.append(Document(page_content=text, metadata=metadata))

        return documents
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


# Combining docs identically
docs = (
    #load_pdf_as_documents("./documents/book1.pdf") +
     load_pdf_as_documents("./documents/book2.pdf")
    #+ load_pdf_as_documents("./documents/book3.pdf")   
)

In [ ]:
len(docs)

In [ ]:
#-----------------------------------CHUNKING--------------------------------
#chunking the documents using Recursive Character text splitter
chunks = RecursiveCharacterTextSplitter(chunk_size =900, chunk_overlap =100).split_documents(docs)

#---------------------------Clean text to avoid UnicodeEncodeError-------------------

for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
len(chunks)

In [ ]:
#-----------------------Indexing------------------------------ 
#(fresh collection each run)
#------------------------------EMBED DOCS-----------------------------------
#Generating Embeddings for given docs using text-embedding-004 from google generative ai embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-001",# or 'models/gemini-embedding-exp-03-07'
    task_type="retrieval_document"# Options: retrieval_query, retrieval_document, semantic_similarity, etc.
)

In [ ]:
#----------------------------------VECTOR STORE-----------------------------------
#Storing in vector database using  FAISS db

import time
# NEW SETTINGS: Process 100 chunks at once, then pause for 60 seconds.
# This pushes exactly 100 chunks to the API per minute, staying perfectly under the limit.
BATCH_SIZE = 100
SLEEP_TIME_SECONDS = 40 
vector_store = None

print(f"Starting vector database creation for {len(chunks)} text chunks...")

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    
    if vector_store is None:
        # Create the initial vector database using the first chunk batch
        vector_store = FAISS.from_documents(batch, embeddings)
    else:
        # Add subsequent chunks to the active store instance
        vector_store.add_documents(batch)
        
    current_chunk_count = min(i + BATCH_SIZE, len(chunks))
    print(f"Embedded [{current_chunk_count}/{len(chunks)}] chunks.")
    
    # Only pause if there are still more chunks left to process
    if current_chunk_count < len(chunks):
        print(f"Pausing for {SLEEP_TIME_SECONDS} seconds to completely reset the 100 RPM quota...")
        time.sleep(SLEEP_TIME_SECONDS)

print("\n Success! All text chunks embedded and vector store created.")


In [ ]:
#-------------------------RETRIEVER-----------------------------
#Creating a retriever using vector store
retriever =  vector_store.as_retriever(search_type='similarity',search_kwargs ={"k":4})

In [ ]:
#---------------------------LLM-------------------------------
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
#llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash",temperature = 0)

In [ ]:
#-------------------- Graph State ---------------------------------------
class State(TypedDict):
    question: str
    
    #doc related
    docs: List[Document]

    #check if docs are good - related to evaluator
    good_docs : List[Document]
    verdict : str
    reason : str
    
    #refining the context
    strips : List[str]# output of decomposition (sentence strips)
    kept_strips : List[str] # after filtering (kept sentences)
    refined_text : str # recomposed internal knowledge (joined kept_strips)
    
    answer: str

In [ ]:
#------------------------------NODES DEFINITION---------------------------------------

#----------------------------------1. RETRIEVER -------------------------------------
def retrieve(state:State) -> State:
    query = state["question"]
    return {"docs": retriever.invoke(query)}

In [ ]:
#threshold for scoring the docs for the evaluator node
UPPER_TH = 0.7
LOWER_TH = 0.3

In [ ]:
#------------------------------2. DOCUMENT EVALUATOR-----------------------------------

# Score-based doc evaluator
class DocEvalScore(BaseModel):
    score : float  = Field(description = "Scoring docs if they are relevant to  the question asked")
    reason : str   = Field(description = "reason for the doc score based on relevance")

doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 1.0].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Also return a short reason.\n"
            "Output JSON only.",
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}"),
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

def eval_each_doc(state:State) -> State:
    query = state['question']

    scores : List[float] = []
    reasons: List[str] = []
    good : List[str] = []

    for d in state['docs']:
        output = doc_eval_chain.invoke({"question": query, "chunk": d.page_content})
        scores.append(output.score)
        reasons.append(output.reason)
        #a. for CORRECT case we will refine only docs with score > LOWER_TH
        if output.score > LOWER_TH:
            good.append(d)
    
    #b. CORRECT if at least one doc > UPPER_TH
    if any(s > UPPER_TH for s in scores):
        return {
            "good_docs": good,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {UPPER_TH}.",
        }

    #c. INCORRECT if all docs < LOWER_TH
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        why = "No chunk was sufficient."
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"All retrieved chunks scored < {LOWER_TH}. {why}",
        }
    #d. Anything in between => AMBIGUOUS
    why = "Mixed relevance signals."
    return {
        "good_docs": good,
        "verdict": "AMBIGUOUS",
        "reason": f"No chunk scored > {UPPER_TH}, but not all were < {LOWER_TH}. {why}",
    }
    

In [ ]:
#-------------------------------3. REFINING DOCS ------------------------------------

#----------------------------HELPER functions -------------------------------

# Sentence-level DECOMPOSER
def decompose_to_sentences(text:str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]


# FILTER (LLM as judge) 

#for structured output using pydantic model 
class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "Return keep=true only if the sentence directly helps answer the question.\n"
            "Use ONLY the sentence. Output JSON only.",
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}"),
    ]
)
    
filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

#-------------------REFINING (Decompose -> Filter -> Recompose)------------------
def refine(state:State) -> State :
    query = state['question']
    # Combining all retrieved documents into one context string
    context = "\n\n".join(d.page_content for d in state['docs']).strip()

    # 1. DECOMPOSITION: context -> sentence strips
    strips = decompose_to_sentences(context)
    
    # 2. FILTERING: keeping only relevant strips
    kept_strips :List[str] = []

    for each in strips:
        if filter_chain.invoke({"question": query, "sentence": each}).keep  :
            kept_strips.append(each)
            
    # 3. RECOMPOSE: join kept strips back together (for internal knowledge)
    refined_context = "\n".join(kept_strips).strip()

    return {
        "strips": strips,
        "kept_strips": kept_strips,
        "refined_context": refined_context,
    }
            
            

In [ ]:
#---------------------------------------4. GENERATE --------------------------------------------

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful ML tutor. Answer ONLY using the provided refined bullets.\n"
            "If the bullets are empty or insufficient, say: 'I don't know based on the provided books.'",
        ),
        ("human", "Question: {question}\n\nRefined context:\n{refined_context}"),
    ]
)

def generate(state: State) -> State:
    output = (answer_prompt | llm).invoke({"question": state["question"], "refined_context": state['refined_context']})
    return {"answer": output.content}

In [ ]:
#-------------------------------5. FAIL---------------------------------------
def fail_node(state: State) -> State:
    return {"answer": f"FAIL: {state['reason']}"}

In [ ]:
#-------------------------------------6. AMBIGIOUS-----------------------------------
def ambiguous_node(state: State) -> State:
    return {"answer": f"Ambiguous: {state['reason']}"}

In [ ]:
#-------------------conditional nodes -------------------------
def route_after_eval(state: State) -> str:
    if state["verdict"] == "CORRECT":
        return "refine"
    elif state["verdict"] == "INCORRECT":
        return "web_search"
    else:
        return "ambiguous"

In [ ]:
#---------------------------------BUILDING GRAPH-------------------------------------
graph = StateGraph(State)

graph.add_node("retrieve", retrieve)
graph.add_node("eval_each_doc",eval_each_doc)
graph.add_node("refine",refine)
graph.add_node("generate", generate)
graph.add_node("fail",fail_node)
graph.add_node("ambiguous",ambiguous_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve","eval_each_doc")
graph.add_conditional_edges("eval_each_doc",route_after_eval,{"refine":"refine","web_search":"fail","ambiguous":"ambiguous"})

graph.add_edge("refine","generate")
graph.add_edge("generate", END)
graph.add_edge("fail",END)

app = graph.compile()

app

In [ ]:
#-------------------------------------Run-----------------------------------------

res = app.invoke(
    {
        "question": "What are attention mechanisms and why are they important in current models?",
        "docs": [],
        "good_docs": [],
        "verdict": "",
        "reason": "",
        "strips": [],
        "kept_strips": [],
        "refined_context": "",
        "answer": "",
    }
)

print("VERDICT:", res["verdict"])
print("REASON:", res["reason"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
# Bias variance tradeoff
# AI news from last week
# What are attention mechanisms and why are they important in current models?